# 02 · Quantization: from 32-bit floats to 8-bit integers

**Goals of this notebook**

1. Understand how computers store numbers, and why 8-bit integers are attractive on a microcontroller.
2. Learn the quantization formula $r \approx s\,(q - z)$ and compute it by hand.
3. Quantize the weights of the baseline model and measure the error.
4. Quantize the *whole* network, so that it only computes with integers, exactly like the Arduino will.
5. Push the bit width down to 4, 3 and 2 bits, and rescue the accuracy with **quantization-aware training**.

In [ ]:
import sys, pathlib, copy
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import matplotlib.pyplot as plt

from aiedge.data import load_gesture_data
from aiedge.models import MLP, count_params
from aiedge.training import train, accuracy, set_seed
from aiedge.quantization import (QuantMLP, asymmetric_params, int8_forward, int8_accuracy,
                                 int8_model_bytes)
from aiedge import plots

First we re-create the data and the baseline from notebook 01. Same seed, so it is the same model; this takes a few seconds.

In [ ]:
data = load_gesture_data()
set_seed(0)
baseline = MLP(hidden=(64, 32))
train(baseline, data, augment=True, verbose=False)
base_acc = accuracy(baseline, data.X_test, data.y_test)
print(f"baseline (float32): test accuracy {base_acc:.1%}, {count_params(baseline):,} parameters "
      f"= {4 * count_params(baseline) / 1024:.1f} KB")

## 1. How computers store numbers

**float32** (a 32-bit floating-point number, `float` in C) can store values like 0.000123 or 45 678.9. It uses 32 bits: 1 sign bit, 8 bits for an exponent (*where is the decimal point?*) and 23 bits for the digits (the *mantissa*). That is very flexible, but it takes 4 bytes, and floating-point arithmetic needs complex hardware.

**int8** (an 8-bit integer, `int8_t` in C) can only store the whole numbers from **-128 to 127**: 256 different values in 1 byte.

Why bother with int8 on a microcontroller?

1. **4x less memory**: 1 byte instead of 4 per weight. This is the big one for us.
2. **Less energy.** Measured on a 45 nm chip (Horowitz, ISSCC 2014):

   | operation | energy |
   |---|---|
   | 8-bit integer add | 0.03 pJ |
   | 32-bit float add | 0.9 pJ |
   | 8-bit integer multiply | 0.2 pJ |
   | 32-bit float multiply | 3.7 pJ |
   | read 32 bits from a small on-chip memory | ~5 pJ |

   Integer maths is ~20x cheaper, and moving fewer bytes around saves energy too.
3. **Speed**: on chips *without* an FPU, float maths is emulated in software and is 10-100x slower. Our Cortex-M4F *has* an FPU, so the gain there comes from special instructions that do two or four 8/16-bit multiplications at once (used by libraries like CMSIS-NN). In notebook 05 you will measure what *our* simple C code achieves.

The question is: **how do we represent numbers like 0.0372 or -1.25 with integers from -128 to 127, without ruining the network?**

## 2. The quantization formula

We pick a **scale** $s$ (a small positive float) and a **zero-point** $z$ (an integer). A real number $r$ is represented by the integer

$$q = \text{clamp}\big(\text{round}(r / s) + z,\ q_{min},\ q_{max}\big)$$

and we can get back (approximately) the real number with

$$r \approx s \cdot (q - z)$$

Think of it as a ruler with 256 marks spaced $s$ apart: every real number is rounded to the nearest mark. The *rounding error* is at most $s/2$, half the distance between marks. The *clamp* cuts off numbers outside the ruler (*clipping error*).

### Symmetric quantization (for weights)
Weights are spread around 0 (both signs). We use $z = 0$ and pick $s$ so that the largest absolute weight lands on 127:

$$s = \frac{\max |w|}{127}$$

Let's do it by hand for four weights:

In [ ]:
w = torch.tensor([-0.62, 0.047, 0.313, 1.27])
s = w.abs().max() / 127                      # scale
q = torch.round(w / s).clamp(-127, 127)      # quantize   -> integers
w_back = s * q                               # dequantize -> approximately w

print(f"scale s = {s:.4f}")
print("original   :", w.numpy())
print("integers q :", q.int().numpy())
print("dequantized:", w_back.numpy().round(4))
print("error      :", (w_back - w).numpy().round(4), f"  (max possible: s/2 = {s / 2:.4f})")

### Asymmetric quantization (for activations)
After a ReLU all values are ≥ 0. Symmetric quantization would waste the whole negative half of the ruler (-128…-1). Instead we map the actual range $[lo, hi]$ onto $[-128, 127]$:

$$s = \frac{hi - lo}{255}, \qquad z = -128 - \text{round}(lo / s)$$

For a ReLU output with range $[0, 6]$: $s = 6/255 \approx 0.0235$ and $z = -128$. So real 0 ↦ -128, real 6 ↦ 127. Note that real **0 is represented exactly** (by $q = z$): this matters, because ReLU produces lots of exact zeros.

In [ ]:
scale, zp = asymmetric_params(0.0, 6.0)
a = torch.tensor([0.0, 0.01, 1.0, 3.3, 6.0])
q = torch.clamp(torch.round(a / scale) + zp, -128, 127)
print(f"scale = {scale:.4f}, zero-point = {zp}")
print("real       :", a.numpy())
print("integers q :", q.int().numpy())
print("dequantized:", (scale * (q - zp)).numpy().round(4))

### The range trade-off
The range must be chosen well:
* **too wide** (e.g. one huge outlier weight) → $s$ is large → all the *normal* values get coarse rounding;
* **too narrow** → large values are clipped.

This is why we often use one scale **per output neuron** ("per-channel") for weights instead of one scale for a whole layer ("per-tensor"): one neuron with large weights then cannot ruin the precision of all the others.

## 3. Quantizing the baseline's weights

Let's look at the weights of the first layer (19 200 numbers):

In [ ]:
W0 = baseline.layers[0].weight.detach()
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.3))
a1.hist(W0.flatten().numpy(), bins=100, color=plots.PALETTE[0])
a1.set_xlabel("weight value"); a1.set_ylabel("count"); a1.set_title("Layer 0: all weights")
a2.bar(range(W0.shape[0]), W0.abs().amax(dim=1).numpy(), color=plots.PALETTE[0])
a2.set_xlabel("neuron (output channel)"); a2.set_ylabel("max |w|"); a2.set_title("Largest weight per neuron")
fig.tight_layout(); plt.show()

Most weights are small, and the largest weight differs quite a lot from neuron to neuron. Let's write a function that quantizes the weights of every layer (per-tensor or per-channel) and puts the *dequantized* values back, so we can measure what the rounding does to the accuracy. This is called **simulated quantization**: still floats, but only values that an integer could represent.

In [ ]:
def quantize_weights(model, bits=8, per_channel=True):
    """Return a copy of `model` whose weights are rounded to `bits`-bit integers (and back)."""
    qmax = 2 ** (bits - 1) - 1                         # 127 for 8 bits, 7 for 4 bits
    m = copy.deepcopy(model)
    with torch.no_grad():
        for layer in m.layers:
            w = layer.weight
            if per_channel:
                max_abs = w.abs().amax(dim=1, keepdim=True)   # one scale per neuron (row)
            else:
                max_abs = w.abs().max()                       # one scale for the layer
            s = max_abs / qmax
            layer.weight.copy_(torch.clamp(torch.round(w / s), -qmax, qmax) * s)
    return m

print(f"{'bits':>5} {'per-tensor':>12} {'per-channel':>12}   (test accuracy, float32 = {base_acc:.1%})")
for bits in [8, 6, 4, 3, 2]:
    acc_t = accuracy(quantize_weights(baseline, bits, per_channel=False), data.X_test, data.y_test)
    acc_c = accuracy(quantize_weights(baseline, bits, per_channel=True), data.X_test, data.y_test)
    print(f"{bits:>5} {acc_t:>12.1%} {acc_c:>12.1%}")

**Observations**
* With **8 bits the accuracy does not change** (differences of a few tenths of a point are noise): the rounding errors are far smaller than what the network cares about. This is why int8 is the industry standard.
* Down to ~4 bits things still work; at 2 bits (only the values -1, 0, +1 times a scale!) the network breaks.
* Down to 4 bits, per-tensor and per-channel perform about the same; at very low bit widths **per-channel is clearly better**.

## 4. Quantizing the whole network: activations too

So far only the *weights* were integers. For the board we want **integer-only inference**: the inputs and the activations between the layers must be int8 as well. For weights we know the range (we can just look at them). Activations depend on the input, so we must **measure** their range by running real data through the network. That is called **calibration**.

`QuantMLP` (in `aiedge/quantization.py`) wraps our model and simulates integer inference:
* input → quantize with (scale, zero-point) of the input;
* each layer → weights quantized per channel, bias quantized to int32;
* after each ReLU → quantize the activation with its own (scale, zero-point).

This approach (train in float, then calibrate and quantize) is called **post-training quantization (PTQ)**.

In [ ]:
ptq = QuantMLP(copy.deepcopy(baseline), weight_bits=8, act_bits=8)
ptq.calibrate(data.X_train)        # measure activation ranges on the training data

for k in range(len(ptq.act_lo)):
    scale, zp = ptq.act_params(k)
    name = "input" if k == 0 else f"after hidden layer {k}"
    print(f"{name:22s} range [{ptq.act_lo[k]:6.2f}, {ptq.act_hi[k]:6.2f}]  ->  scale {scale:.4f}, zero-point {zp}")

ptq_acc = accuracy(ptq, data.X_test, data.y_test)
print(f"\nPTQ int8 (simulated) test accuracy: {ptq_acc:.1%}   (float32: {base_acc:.1%})")

## 5. How does a layer compute with integers only?

Write each real number as $r = s\,(q - z)$. For one neuron $j$ of a layer, with input scale $s_x$, zero-point $z_x$, weight scale $s_{w,j}$ (and weight zero-point 0):

$$y_j = \sum_i w_{ji} x_i + b_j = \sum_i s_{w,j} q_{w,ji} \cdot s_x (q_{x,i} - z_x) + b_j = s_{w,j} s_x \underbrace{\Big[\, q_{b,j} + \sum_i q_{w,ji}(q_{x,i} - z_x) \Big]}_{\text{acc}_j\text{: pure integer maths!}}$$

where the bias is stored as a 32-bit integer $q_{b,j} = \text{round}\big(b_j / (s_{w,j} s_x)\big)$. The expensive part, the sum with all the MACs, uses only integer multiplications and additions. Then we convert to the output's int8 representation:

$$q_{y,j} = \text{clamp}\big(\text{round}(M_j \cdot \text{acc}_j) + z_y,\ z_y,\ 127\big) \qquad M_j = \frac{s_{w,j}\, s_x}{s_y}$$

Two neat details:
* The lower clamp at $z_y$ **is the ReLU**: $z_y$ is where real 0 lies, so anything below it was negative.
* $M_j$ is one float multiplication per *neuron*, not per weight, so it costs almost nothing. (Chips without an FPU replace it by an integer multiplication plus a bit shift.)

The accumulator must be 32 bits wide: 300 products of up to 127 × 255 add up to millions.

`int8_forward` implements exactly these formulas with NumPy integers. It is the twin of the C++ code that will run on the Arduino (notebook 05 checks that both give *identical* results).

In [ ]:
int8_model = ptq.to_int8()
L0 = int8_model["layers"][0]
print("layer 0 weights :", L0["w_q"].dtype, L0["w_q"].shape, " e.g.", L0["w_q"][0, :8])
print("layer 0 biases  :", L0["b_q"].dtype, L0["b_q"].shape, " e.g.", L0["b_q"][:4])
print("layer 0 M       :", L0["multiplier"].dtype, " e.g.", L0["multiplier"][:3])

logits_int = int8_forward(int8_model, data.X_test.numpy())
with torch.no_grad():
    logits_sim = ptq(data.X_test).numpy()
print(f"\ninteger-only accuracy: {int8_accuracy(int8_model, data.X_test, data.y_test):.1%}")
print(f"same predictions as the simulation: {(logits_int.argmax(1) == logits_sim.argmax(1)).mean():.1%}"
      f"   (largest logit difference {np.abs(logits_int - logits_sim).max():.1e})")
print(f"memory: {int8_model_bytes(int8_model) / 1024:.1f} KB  (float32: {4 * count_params(baseline) / 1024:.1f} KB)")

So the integer-only network is (practically) **as accurate as the float network, at ~1/4 of the memory**. The few extra bytes on top of a pure 1/4 come from the int32 biases and one float multiplier per neuron.

## 6. Fewer bits and quantization-aware training (QAT)

At 8 bits, PTQ is enough. But what if we want 4, 3 or even 2 bits? We saw that PTQ breaks down there. The fix: let the network **experience the rounding during training**, so it can learn weights that are robust to it.

**Quantization-aware training** (QAT):
* In the **forward pass** we use *fake quantization*: weights and activations are quantized and immediately dequantized, so the network sees exactly the rounding errors it will see on the device.
* In the **backward pass** there is a problem: `round()` is a staircase, so its gradient is 0 almost everywhere, and nothing would be learned. The trick is the **straight-through estimator (STE)**: during backpropagation we *pretend* the rounding was not there and pass the gradient straight through. In PyTorch this is one line: `x + (fake_quant(x) - x).detach()`.
* The underlying float weights are updated normally; they "drift" to values that still work well after rounding.

We start from the trained baseline and fine-tune for a few epochs with fake quantization. To be fair, we also fine-tune the *float* model for the same number of epochs (the "control"), so we can see what part of the improvement is just extra training. In all cases the activations stay 8-bit.

In [ ]:
bit_widths = [8, 4, 3, 2]
results = {"PTQ": [], "QAT": []}
set_seed(0)
control = copy.deepcopy(baseline)
train(control, data, lr=5e-4, epochs=30, patience=10, augment=True, verbose=False)
control_acc = accuracy(control, data.X_test, data.y_test)

for bits in bit_widths:
    q = QuantMLP(copy.deepcopy(baseline), weight_bits=bits)
    q.calibrate(data.X_train)
    results["PTQ"].append(accuracy(q, data.X_test, data.y_test))
    train(q, data, lr=5e-4, epochs=30, patience=10, augment=True, verbose=False)   # <- this is QAT
    results["QAT"].append(accuracy(q, data.X_test, data.y_test))
    print(f"{bits} bits: PTQ {results['PTQ'][-1]:.1%}   QAT {results['QAT'][-1]:.1%}")
print(f"float32 baseline {base_acc:.1%}, float32 fine-tuned (control) {control_acc:.1%}")

In [ ]:
ax = plots.plot_curves(bit_widths, results, xlabel="weight bits", title="Accuracy vs weight bit width")
ax.axhline(base_acc, color=plots.MUTED, ls=":", lw=1.5)
ax.text(bit_widths[-1], base_acc, "  float32 baseline", va="bottom", color=plots.MUTED)
ax.invert_xaxis(); plt.show()

**What do you see?**
* At **8 and 4 bits**, PTQ, QAT and the float **control** are all within about a point of each other: noise (remember notebook 01!). Any small advantage of QAT over PTQ here could just as well come from the extra training epochs. That's what the control measures. For int8, PTQ is all you need.
* At **2 bits**, PTQ collapses, while QAT brings the accuracy back to within about a point of the float models. The network has *learned* to live with only 3 different weight values per neuron (−1, 0, +1 times a scale).

This is why we included the control. Without it we might have concluded that QAT improves 8-bit models!

### Memory summary

In [ ]:
n_w = sum(l.weight.numel() for l in baseline.layers)
n_b = sum(l.bias.numel() for l in baseline.layers)
print(f"{'format':<34}{'weight bytes':>13}")
print(f"{'float32':<34}{4 * (n_w + n_b):>13,}")
for bits in [8, 4, 2]:
    # biases (int32) + one float multiplier per neuron stay 32-bit
    label = f"int{bits} weights" + (" (2 per byte)" if bits == 4 else " (4 per byte)" if bits == 2 else "")
    print(f"{label:<34}{n_w * bits // 8 + 8 * n_b:>13,}")

Note that 4- and 2-bit weights only save memory if you **pack** several of them into one byte, and then the C code must unpack them again (a bit of extra work per weight). The Arduino code in this course uses int8: the standard choice, supported directly by the hardware.

## Summary

* Quantization maps real numbers to integers: $q = \text{round}(r/s) + z$, $r \approx s\,(q - z)$.
* Weights: symmetric, one scale per neuron (per-channel). Activations: asymmetric, ranges found by **calibration**.
* With the right formulas a whole layer runs on **integer MACs** plus one float multiply per neuron.
* **int8 PTQ** shrinks the model 4x at essentially no accuracy cost.
* At very low bit widths, **QAT** (fake quantization + straight-through estimator) recovers the accuracy.

## Exercises

1. ⭐ **By hand.** Quantize the weights `[0.9, -0.3, 0.02, -1.8]` to 4 bits (symmetric, $q \in [-7, 7]$) on paper. What are $s$, the $q$'s, the dequantized values and the largest error? Check with code.
2. ⭐ **Outliers.** Take a copy of the baseline and set *one* weight of layer 0 to 20.0. Compare per-tensor vs per-channel 4-bit accuracy (`quantize_weights`). Explain the result with the "ruler" picture.
3. **Activation bits.** Try `QuantMLP(..., act_bits=4)` with weights at 8 bits. (Note: `to_int8()` only supports 8-bit activations, but the simulation works.) Which is more sensitive, weights or activations?
4. **Calibration data.** Calibrate the PTQ model on only the first 10 training examples instead of all. What happens to the ranges and the accuracy? Why?
5. **Smaller models.** Repeat the bit-width experiment for a small `MLP(hidden=(16, 8))`. Do small networks tolerate quantization better or worse? Why could that be?